# SegmentAnyTree — Quick Start

This notebook runs tree instance segmentation on your point cloud files using the pre-trained PointGroup-PAPER model.

**Input**: `.las`, `.laz`, or `.ply` files in `/data/input/`  
**Output**: Segmented `.las` files in `/data/output/final_results/`

Reference: [Wielgosz et al. (2024)](https://doi.org/10.1016/j.rse.2024.114367)

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Verify input files exist
from pathlib import Path

input_dir = Path("/data/input")
output_dir = Path("/data/output")

files = list(input_dir.glob("*.las")) + list(input_dir.glob("*.laz")) + list(input_dir.glob("*.ply"))
print(f"Found {len(files)} input files:")
for f in files[:10]:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")
if len(files) > 10:
    print(f"  ... and {len(files) - 10} more")
if not files:
    print("No input files found! Mount your data to /data/input/")

## Run Inference

This runs the full pipeline: coordinate transform → model inference → merge results.

In [ ]:
import subprocess, os
os.environ["SAT_ROOT"] = "/opt/segmentanytree"

result = subprocess.run(
    ["bash", "scripts/run_inference.sh", str(input_dir), str(output_dir), "true"],
    cwd=os.environ["SAT_ROOT"],
    capture_output=False,
    text=True
)
print(f"Exit code: {result.returncode}")

## Check Results

In [ ]:
results_dir = output_dir / "final_results"
results = list(results_dir.glob("*")) if results_dir.exists() else []
print(f"Output files ({len(results)}):")
for f in results:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

if results:
    # Preview first result
    import laspy
    las = laspy.read(str(results[0]))
    print(f"\nPreview: {results[0].name}")
    print(f"  Points: {len(las.points):,}")
    print(f"  Dimensions: {list(las.point_format.dimension_names)}")
    if 'PredInstance' in las.point_format.dimension_names:
        n_trees = len(set(las.PredInstance)) - (1 if 0 in las.PredInstance else 0)
        print(f"  Detected trees: {n_trees}")